# SpamJam: modelling

SpamJam flags spam (`is_spam` = 1) for GrandPied's IT team. The problem is described in the README, and the data is explored and split in `data_exploration.ipynb`. Here we build and compare models on the training set.

**What guides our choices.** For GrandPied, a false positive (a client email flagged as spam) costs more than a false negative (spam that gets through). So:
- **Metric:** we compare models with average precision, then pick a decision threshold on out-of-fold predictions that keeps precision on spam high.
- **Protocol:** every choice is made on the training set with cross-validation; the test set is used once, at the end.

**Scope.** This is a proof of concept on Spambase, whose legitimate emails all come from one person. GrandPied would retrain the model on its own emails before real use.

## Preprocessing pipeline

Everything below runs inside a scikit-learn `Pipeline`, so each CV fold learns its preprocessing from its training part only.

- **Capital-letter counts:** log transform, then scaling. They are very skewed (see `data_exploration.ipynb`), and the log stops a few extreme emails from dominating.
- **All other features:** scaling only.
- **Scaling:** `StandardScaler`, because k-NN, the linear SVM and regularised logistic regression depend on feature scale. `MinMaxScaler` is crushed by outliers, and `RobustScaler` can't scale features whose quartiles are all 0.
- **No imputation:** there are no missing values; `predict` will reject incomplete input instead of guessing.
- **No encoding:** all features are numeric.

**How we decide.** We compare options with the same 5-fold CV and average precision. An option counts as better only if it beats the other by more than one standard error of the CV score; otherwise we keep the simpler one.

## Log transform and L1 vs L2

In [ ]:
import numpy as np
import pandas as pd
import yaml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import FunctionTransformer

from spamjam.constants import CONFIG_PATH
from spamjam.data import DatasetLoader
from spamjam.features import build_pipeline

loader = DatasetLoader.from_config()
train_df, _ = loader.load_split()   # test set not used until the end

with open(CONFIG_PATH, encoding="utf-8") as f:
    config = yaml.safe_load(f)
donor = config["features"]["donor_columns"]

X_train = train_df.drop(columns="is_spam")
y_train = train_df["is_spam"]

grid = {
    "preprocess__capital__log": [
        FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
        "passthrough",                      # = no log transform
    ],
    "model__penalty": ["l1", "l2"],
    "model__C": [0.01, 0.1, 1, 10],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config["random_state"])
search = GridSearchCV(
    build_pipeline(LogisticRegression(solver="liblinear")),
    grid, cv=cv, scoring="average_precision", n_jobs=-1,
).fit(X_train, y_train)

In [ ]:
results = pd.DataFrame(search.cv_results_)

results["log"] = results["param_preprocess__capital__log"].map(
    lambda v: "no" if isinstance(v, str) else "yes")
cols = ["log", "param_model__penalty", "param_model__C",
        "mean_test_score", "std_test_score"]
best = results.loc[results.groupby(["log", "param_model__penalty"])
                   ["mean_test_score"].idxmax(), cols]
best.sort_values("mean_test_score", ascending=False).round(4)

**Decisions.**
- **Log transform: keep it.** With the log, the best CV average precision is about 0.954; without it, about 0.944. The gain (~0.010) is about two standard errors, so it is a real improvement.
- **L1 vs L2: L1.** The best L1 (C=1) and L2 (C=10) scores differ by about 0.001, well within one standard error. Following our rule, we keep the simpler model: L1, which also sets weak features to zero.

## Donor-specific words

Some words describe the dataset's donor (George, at HP Labs) rather than spam: `george`, `hp`, `hpl`, `650`, `857`, `415`, `labs`, `telnet`, `technology`. They would mean nothing for GrandPied's emails, and several are strongly correlated (`857` and `415`: 0.994). We run the same search without them and compare the best scores.

Note that cross-validation can't fully answer this question: its validation folds also come from the same donor, so these words look useful in CV even though they would mean nothing for GrandPied.

In [ ]:
search_dropped = GridSearchCV(
    build_pipeline(LogisticRegression(solver="liblinear")),
    grid, cv=cv, scoring="average_precision", n_jobs=-1,
).fit(X_train.drop(columns=donor), y_train)

std_best = search.cv_results_["std_test_score"][search.best_index_]
one_se = std_best / np.sqrt(cv.get_n_splits())

print(f"All features:        {search.best_score_:.4f}")
print(f"Without donor words: {search_dropped.best_score_:.4f}")
print(f"Difference:          {search.best_score_ - search_dropped.best_score_:.4f}")
print(f"One standard error:  {one_se:.4f}")

**Decision: keep the donor-specific words.** Dropping them lowers the best CV average precision from about 0.954 to about 0.938 (a drop of about 0.016, roughly three standard errors), so on Spambase they clearly help. But that gain is exactly the part that would not carry over to GrandPied: CV tests on emails from the same donor, so it can't measure it.

We keep them because our model is a proof of concept on Spambase, and we report this as a limitation instead:
- our scores are likely **optimistic** for GrandPied's emails;
- the legitimate emails all come from one person, so other features (e.g. `meeting`, `project`, `edu`, `1999`) may also partly reflect his work rather than ham in general;
- before real use, the model must be **retrained on GrandPied's own labelled emails**, with the same pipeline.